# Notebook 05 — Adaptive Scoring & Temporal Decision-Support **Prototype**
**J26-IT-388 | Component 4 | PP1 | Frozen Dreaddit model**

**Scope:** technical demonstration only. Dreaddit posts are independent annotated text segments; they are **not** longitudinal data about any one person. All time series, synthetic case identifiers, timestamps, thresholds and event labels below are **simulated**. A stress-related-text classifier score is **not a clinical risk probability**. We do not estimate diagnosis, intervention effectiveness or changes in any person's health. No model training or tuning occurs in this notebook.

**Outputs:** auditable synthetic sequences, static-vs-smoothed demonstrations, missing/staleness handling, tests, runtime, plots, PP1 manifest and limitations.

## 1 — Setup and reproducibility
Run from a Jupyter notebook in your project’s `notebooks/` folder. Required packages: `pandas numpy scikit-learn joblib matplotlib`. Reuse the installed `.venv` environment from earlier notebooks. The notebook **does not** overwrite the saved classifier.

In [ ]:
from pathlib import Path
import sys, json, hashlib, time, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

SEED = 42
rng = np.random.default_rng(SEED)
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "models" / "dreaddit_tfidf_logreg.joblib").exists():
    raise FileNotFoundError("Run from repository root or notebooks/ directory")
MODEL_PATH = ROOT / "models" / "dreaddit_tfidf_logreg.joblib"
TEST_PATH = ROOT / "data" / "raw" / "dreaddit" / "dreaddit-test.csv"
OUT_METRICS = ROOT / "experiments" / "metrics" / "notebook05"
OUT_FIGURES = ROOT / "experiments" / "figures" / "notebook05"
for folder in (OUT_METRICS, OUT_FIGURES):
    folder.mkdir(parents=True, exist_ok=True)
assert MODEL_PATH.exists(), f"Missing frozen model: {MODEL_PATH}"
assert TEST_PATH.exists(), f"Missing Dreaddit test CSV: {TEST_PATH}"
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()
MODEL_HASH_START = sha256_file(MODEL_PATH)
print("Repository:", ROOT)
print("Frozen model SHA256:", MODEL_HASH_START)
print("Packages:", {"python":platform.python_version(), "sklearn":sklearn.__version__, "numpy":np.__version__, "pandas":pd.__version__})

## 2 — Read-only inference from the frozen classifier
Load only the model artifact created in your trusted local research repository (joblib is an executable serialization format). Use its existing `predict_proba` function. The held-out test texts supply plausible **score values only**, never longitudinal histories.

In [ ]:
pipeline = joblib.load(MODEL_PATH)  # trusted artifact from Notebook 02 ONLY
assert hasattr(pipeline, "predict_proba"), "Expected existing probability prediction interface"
test = pd.read_csv(TEST_PATH)
assert {"text","label"}.issubset(test.columns)
assert test["text"].notna().all()
classes = list(pipeline.classes_)
assert set(map(int,classes)) == {0,1}, f"Unexpected classes: {classes}"
positive_idx = classes.index(1)
t0 = time.perf_counter()
p = np.asarray(pipeline.predict_proba(test["text"].astype(str).tolist()))[:, positive_idx]
inference_seconds = time.perf_counter()-t0
assert len(p) == len(test) and np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
print("Held-out examples:",len(p), "Probability min/median/max:",np.round([p.min(),np.median(p),p.max()],4))
print("Frozen inference batch seconds:",round(inference_seconds,4))

## 3 — Construct **synthetic** timestamped scenarios
For repeatability, choose held-out predictions by probability rank, then rearrange distinct examples to fabricate sequences. **All timestamps, pseudonymous scenario IDs and sequence order are fictional.** Some scenarios instead use explicitly hand-written numeric stress-test values. We export sample indices and scores only—**no raw post text or personally identifying attributes**.

In [ ]:
ranked = np.argsort(p, kind="stable")
# select distinct held-out rows across the model-score range
quantiles = np.linspace(0.05, 0.95, 9)
chosen = [int(ranked[min(int(round(q*(len(ranked)-1))),len(ranked)-1)]) for q in quantiles]
assert len(set(chosen)) == len(chosen)
low_to_high = np.asarray([p[i] for i in chosen], dtype=float)
base_time = pd.Timestamp("2026-10-01T12:00:00Z")
rows=[]
def add_scenario(name, probabilities, origins, missing_at=()):
    assert len(probabilities)==len(origins)
    for k, (pr,origin) in enumerate(zip(probabilities,origins)):
        rows.append({"scenario":name,"synthetic_step":k,
                     "synthetic_timestamp":base_time+pd.Timedelta(days=k),
                     "model_probability":np.nan if k in missing_at else float(pr),
                     "heldout_sample_index":origin,"synthetic":True})
add_scenario("increasing_model_scores",low_to_high,chosen)
add_scenario("decreasing_model_scores",low_to_high[::-1],chosen[::-1])
order=[1,7,2,6,3,8,0,5,4]
add_scenario("oscillating_model_scores",low_to_high[order],[chosen[i] for i in order])
# Repeated use of SAME model result is explicitly simulated; not new observations
add_scenario("constant_model_score",[float(low_to_high[4])]*9,[chosen[4]]*9)
add_scenario("missing_model_scores",low_to_high,chosen,missing_at={2,3,4,7})
# stress-test inputs are invented numeric probabilities, NOT outputs from real examples
add_scenario("manual_shock_test",[.2,.2,.2,.95,.95,.2,.2,.2,.2],[None]*9)
add_scenario("manual_all_missing",[.5]*9,[None]*9,missing_at=set(range(9)))
seq=pd.DataFrame(rows).sort_values(["scenario","synthetic_step"]).reset_index(drop=True)
assert seq["synthetic"].all() and seq.groupby("scenario")["synthetic_step"].apply(lambda s:list(s)==list(range(9))).all()
seq.to_csv(OUT_METRICS/"synthetic_input_sequences.csv",index=False)
print(seq.groupby("scenario",sort=True)["model_probability"].agg(["count","min","max"]).to_string())

## 4 — Declare the scoring algorithm (design choice, not validated risk model)

**Static snapshot comparator:** show the latest incoming classifier probability `p_t` without using history. It is a *point-in-time* baseline, **not** a calibrated mental-health risk score.

**Experimental adaptive score:** an exponentially weighted moving average, `s_t = α p_t + (1−α) s_(t−1)` with illustrative `α = 0.35`. The first valid observation initializes the score. For a missing observation, hold the prior score but mark it *stale*; after more than `MAX_GAP=2` consecutive missing observations the output is unavailable and alerts are suppressed until a fresh observation. To avoid treating stale signals as fresh, the first observed value after the gap **reinitializes** the score (we do not retrospectively impute any missing observation).

Demonstration bands use illustrative 0.35/0.65 boundaries: `lower-score / middle-score / higher-score`. These are **engineering display bands**, not clinically validated thresholds or diagnostic classes. The comparison measures smoothing and lag (a trade-off), not health-prediction improvement.

In [ ]:
ALPHA=0.35
MAX_GAP=2
LOW_BOUND=0.35
HIGH_BOUND=0.65

def band(score):
    if score is None or not np.isfinite(score):
        return "unavailable"
    return "lower-score" if score < LOW_BOUND else ("middle-score" if score < HIGH_BOUND else "higher-score")

def process_one_scenario(group, alpha=ALPHA, max_gap=MAX_GAP):
    if not (0<alpha<=1): raise ValueError("alpha must be in (0,1]")
    if max_gap < 0: raise ValueError("max_gap must be >=0")
    frame=group.sort_values("synthetic_step").copy()
    if frame["synthetic_step"].duplicated().any(): raise ValueError("duplicate time step")
    if frame["synthetic_timestamp"].duplicated().any(): raise ValueError("duplicate timestamp")
    previous=None; misses=0; records=[]
    for _,item in frame.iterrows():
        incoming=item["model_probability"]
        valid=pd.notna(incoming) and np.isfinite(incoming) and 0<=incoming<=1
        if pd.notna(incoming) and not valid: raise ValueError("Invalid non-missing probability")
        if valid:
            # Reinitialize after a long data gap to avoid preserving a stale trajectory
            reset = previous is None or misses > max_gap
            current=float(incoming) if reset else alpha*float(incoming)+(1-alpha)*previous
            previous=current; misses=0
            state="fresh_reinitialized" if reset else "fresh_updated"
            available=True
        else:
            misses+=1
            available=previous is not None and misses<=max_gap
            current=previous if available else np.nan
            state="held_stale" if available else "unavailable_stale"
        current=float(current) if pd.notna(current) else np.nan
        static=float(incoming) if valid else np.nan
        records.append({**item.to_dict(), "static_snapshot":static,
                        "adaptive_score":current,"data_state":state,
                        "consecutive_missing":misses,
                        "display_band":band(current),
                        "alert_eligible":bool(valid and available),
                        "prototype_attention_flag":bool(valid and available and current>=HIGH_BOUND)})
    return pd.DataFrame(records)

def evaluate_all(data, alpha=ALPHA):
    return pd.concat([process_one_scenario(g,alpha=alpha) for _,g in data.groupby("scenario",sort=True)],ignore_index=True)

scored=evaluate_all(seq)
scored.to_csv(OUT_METRICS/"synthetic_scoring_results.csv",index=False)
print(scored.groupby("scenario")[["prototype_attention_flag","alert_eligible"]].sum())

## 5 — Static vs adaptive trend visualization
The graphs compare the *same input classifier probabilities* with a history-dependent smoothing transformation. Blank segments are intentionally absent data. Shaded prototype score bands are illustrative only.

In [ ]:
fig, axs=plt.subplots(3,2,figsize=(14,12),sharex=True)
selected=["increasing_model_scores","decreasing_model_scores","oscillating_model_scores","constant_model_score","missing_model_scores","manual_shock_test"]
for ax,name in zip(axs.ravel(),selected):
    frame=scored[scored.scenario==name].sort_values("synthetic_step")
    ax.plot(frame.synthetic_step,frame.static_snapshot,"o--",label="Static snapshot: latest model p",alpha=.85)
    ax.plot(frame.synthetic_step,frame.adaptive_score,"s-",label="Synthetic smoothed score",alpha=.85)
    missing=frame[frame.model_probability.isna()]
    if len(missing):ax.scatter(missing.synthetic_step, [0.04]*len(missing),marker="x",label="Missing input")
    ax.axhline(LOW_BOUND,ls=":",alpha=.45)
    ax.axhline(HIGH_BOUND,ls=":",alpha=.45)
    ax.set_title(name.replace("_"," "))
    ax.set_ylim(0,1.02);ax.set_ylabel("Illustrative score [0,1]");ax.grid(alpha=.25)
    ax.set_xticks(range(9))
for ax in axs[-1]:ax.set_xlabel("Fictional daily step")
handles, labels=axs[0,0].get_legend_handles_labels()
fig.legend(handles,labels,loc="lower center",ncol=3,bbox_to_anchor=(.5,-.015))
fig.suptitle("Synthetic temporal scenarios: static snapshot vs adaptive smoothing",fontsize=14)
fig.tight_layout(rect=(0,.035,1,.965))
fig.savefig(OUT_FIGURES/"01_static_vs_adaptive_scenarios.png",dpi=170,bbox_inches="tight")
plt.show()

## 6 — Missing-data behavior and alert eligibility
A missing observation is never reclassified as a fresh prediction. The prototype flag triggers **only** on a *fresh* score in the illustrative higher-score band. No alerts are issued for missing or unavailable states, and a flag is **not** a request for medical action.

In [ ]:
focus=scored[scored.scenario=="missing_model_scores"][["synthetic_step","model_probability","static_snapshot","adaptive_score","consecutive_missing","data_state","alert_eligible","prototype_attention_flag"]]
print(focus.to_string(index=False))
assert not scored.loc[scored.model_probability.isna(),"alert_eligible"].any()
assert not scored.loc[scored.model_probability.isna(),"prototype_attention_flag"].any()
assert scored.loc[scored.scenario=="manual_all_missing","adaptive_score"].isna().all()
assert (scored.loc[scored.scenario=="missing_model_scores"].query("consecutive_missing > @MAX_GAP")["data_state"] == "unavailable_stale").all()

## 7 — Comparative *behavior* metrics
These describe how the temporal transformation behaves; they are **not** predictive accuracy, sensitivity or clinical utility. We measure total variation on contiguous observed-pair steps, mean difference from static snapshot, and movement around a synthetic shock. Smoothing can reduce variation **at the cost of delayed response**.

In [ ]:
def comparative_metrics(frame):
    frame=frame.sort_values("synthetic_step").reset_index(drop=True)
    observed=frame.model_probability.notna().to_numpy()
    valid_pairs=observed[1:] & observed[:-1]
    x=frame.static_snapshot.to_numpy(float)
    y=frame.adaptive_score.to_numpy(float)
    valid_difference=observed & np.isfinite(y)
    def mean_pair_abs_diff(values):
        d=np.abs(np.diff(values))
        return float(d[valid_pairs].mean()) if valid_pairs.any() else None
    return {"scenario":frame.scenario.iloc[0], "n_steps":len(frame),
            "n_observed":int(observed.sum()),"n_missing":int((~observed).sum()),
            "static_mean_abs_consecutive_change":mean_pair_abs_diff(x),
            "adaptive_mean_abs_consecutive_change":mean_pair_abs_diff(y),
            "mean_abs_adaptive_vs_static":float(np.abs(y[valid_difference]-x[valid_difference]).mean()) if valid_difference.any() else None,
            "n_prototype_attention_flags":int(frame.prototype_attention_flag.sum()),
            "n_unavailable_steps":int((frame.data_state=="unavailable_stale").sum())}
comparison=pd.DataFrame([comparative_metrics(g) for _,g in scored.groupby("scenario",sort=True)])
comparison.to_csv(OUT_METRICS/"static_adaptive_comparison.csv",index=False)
print(comparison.to_string(index=False))

## 8 — Tests: determinism, range, missing streak, recovery and edge cases
These checks validate engineering behavior only. They do **not** certify the thresholds or score as health-related measurements.

In [ ]:
checks={}
def check(label, condition):
    checks[label]=bool(condition)
    assert bool(condition), f"FAILED: {label}"
check("deterministic_repeat_run", scored.equals(evaluate_all(seq)))
check("all_available_scores_in_unit_interval",scored.adaptive_score.dropna().between(0,1).all())
check("no_flags_without_fresh_input",not scored.loc[scored.model_probability.isna(),"prototype_attention_flag"].any())
check("all_missing_scenario_unavailable",scored.loc[scored.scenario=="manual_all_missing","adaptive_score"].isna().all())
check("no_change_on_identical_fresh_inputs",np.allclose(scored.loc[scored.scenario=="constant_model_score","adaptive_score"],float(low_to_high[4])))
check("increasing_observations_nondecreasing",np.all(np.diff(scored.loc[scored.scenario=="increasing_model_scores","adaptive_score"])>=-1e-12))
check("decreasing_observations_nonincreasing",np.all(np.diff(scored.loc[scored.scenario=="decreasing_model_scores","adaptive_score"])<=1e-12))
miss=scored[scored.scenario=="missing_model_scores"].reset_index(drop=True)
check("long_missing_gap_unavailable",miss.loc[4,"data_state"]=="unavailable_stale" and pd.isna(miss.loc[4,"adaptive_score"]))
check("reappearance_resets_after_gap",miss.loc[5,"data_state"]=="fresh_reinitialized" and np.isclose(miss.loc[5,"adaptive_score"],miss.loc[5,"model_probability"]))
check("initial_missing_do_not_fabricate",scored.loc[scored.scenario=="manual_all_missing","static_snapshot"].isna().all())
# explicit invalid-value rejection (rather than treating invalid input as missing)
bad=seq.query("scenario == 'manual_shock_test'").copy()
bad.loc[bad.index[0],"model_probability"]=1.1
try:
    process_one_scenario(bad)
    raised=False
except ValueError:
    raised=True
check("reject_out_of_range_probability",raised)
# alpha=1 equals static snapshot for valid observations; no smoothing
alpha_one=evaluate_all(seq,alpha=1)
valid=alpha_one.model_probability.notna()
check("alpha_one_reduces_to_latest_observation",np.allclose(alpha_one.loc[valid,"adaptive_score"],alpha_one.loc[valid,"static_snapshot"]))
# alpha sensitivity and tunable responsiveness; not supervised clinical optimisation
alphas={str(a):evaluate_all(seq,alpha=a) for a in (0.2,0.35,0.65)}
shock={a:float(df.query("scenario=='manual_shock_test' and synthetic_step==3").adaptive_score.iloc[0]) for a,df in alphas.items()}
check("higher_alpha_reacts_faster_to_upward_shock",shock["0.2"] < shock["0.35"] < shock["0.65"])
print(f"{sum(checks.values())}/{len(checks)} tests passed")
print("Shock-step example by alpha:",shock)
(OUT_METRICS/"validation_checks.json").write_text(json.dumps({"checks":checks,"passed":sum(checks.values()),"total":len(checks)},indent=2))

## 9 — Score sensitivity + runtime
Varying `α` is a *design sensitivity test*. We do not select a value based on test labels, health outcomes or final clinical impact. Report experimental latency separately from user-facing API latency.

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.5))
for a,df in alphas.items():
    grp=df[df.scenario=="manual_shock_test"].sort_values("synthetic_step")
    ax.plot(grp.synthetic_step,grp.adaptive_score,marker="o",label=f"alpha={a}")
ref=seq[seq.scenario=="manual_shock_test"]
ax.plot(ref.synthetic_step,ref.model_probability,"k--",label="Synthetic input",alpha=.6)
ax.set_ylim(0,1.03);ax.set_xlabel("Fictional daily step");ax.set_ylabel("Illustrative score")
ax.set_title("Smoothing sensitivity: response to an invented shock")
ax.grid(alpha=.2);ax.legend()
fig.tight_layout();fig.savefig(OUT_FIGURES/"02_alpha_sensitivity.png",dpi=170);plt.show()

runtime_iterations=200
t0=time.perf_counter()
for _ in range(runtime_iterations):
    _=evaluate_all(seq)
runtime_total=time.perf_counter()-t0
runtime={"n_synthetic_steps":int(len(seq)),"n_scenarios":int(seq.scenario.nunique()),
         "repetitions":runtime_iterations,"mean_full_batch_seconds":runtime_total/runtime_iterations,
         "mean_ms_per_synthetic_step_estimate":runtime_total/runtime_iterations/len(seq)*1000,
         "warning":"local Python dataframe processing benchmark; not individual endpoint p95 or end-to-end user latency"}
(OUT_METRICS/"processing_runtime.json").write_text(json.dumps(runtime,indent=2))
print(runtime)

## 10 — Generate PP1 manifest and acceptance report
A frozen-model integrity check ensures we did not modify the artifact. The report clearly separates **observed model probabilities from unrelated held-out Reddit texts** and **invented manual probability sequences**. Do not claim user monitoring, clinical validity, classification improvement or longitudinal outcome validation.

In [ ]:
MODEL_HASH_END=sha256_file(MODEL_PATH)
assert MODEL_HASH_START==MODEL_HASH_END,"Frozen model file was changed!"
manifest={
 "project":"J26-IT-388 Component 4", "notebook":"05_adaptive_scoring",
 "run_utc":datetime.now(timezone.utc).isoformat(),
 "task":"proof-of-concept temporal scoring only; no original model training",
 "model_sha256_before":MODEL_HASH_START,"model_sha256_after":MODEL_HASH_END,
 "test_sha256":sha256_file(TEST_PATH),"test_rows":len(test),
 "source_of_scores":"frozen-model held-out predictions rearranged into fictional sequences; manual synthetic shock and all-missing sequences",
 "all_sequences_synthetic":True,"timestamps_real":False,"same_user_observations":False,
 "params":{"alpha":ALPHA,"max_consecutive_missing_before_unavailable":MAX_GAP,
           "illustrative_lower_bound":LOW_BOUND,"illustrative_higher_bound":HIGH_BOUND},
 "comparative_metrics":comparison.replace({np.nan:None}).to_dict(orient="records"),
 "validation":{"passed":int(sum(checks.values())),"total":len(checks)},
 "runtime":runtime,
 "library_versions":{"python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__,"sklearn":sklearn.__version__},
 "limitations":[
  "Dreaddit has no verified per-user longitudinal trajectories; these are fictional sequences",
  "Model outputs concern stress-related text labels, not calibrated clinical mental-health risk",
  "Score smoothing cannot establish early detection or improved predictive accuracy",
  "0.35/0.65 threshold bands and alpha=0.35 are illustrative design choices, not validated clinical cutoffs",
  "Smoothing delays reactions to genuine changes; stale observations suppress flags",
  "Post examples are repurposed from unrelated held-out samples; no real user timeline is implied",
  "No clinician or human usability evaluation, and no real-time API benchmark, has been performed"
 ]}
(OUT_METRICS/"pp1_notebook05_manifest.json").write_text(json.dumps(manifest,indent=2,allow_nan=False,default=str))
summary_text = (
    "# Notebook 05 — PP1 evidence summary\n\n"
    + f"Frozen-model SHA256: `{MODEL_HASH_END}`\n\n"
    + f"Synthetic scenarios: **{seq.scenario.nunique()}**, synthetic steps: **{len(seq)}**, checks passed: **{sum(checks.values())}/{len(checks)}**.\n\n"
    + "**What was shown:** smoothing vs static snapshots, missing/stale-data handling, reproducibility, sensitivity and local runtimes.\n\n"
    + "**What was NOT shown:** clinical risk prediction, actual changes in an individual's mental health, validated clinical thresholds, or better classification performance.\n"
)
(OUT_METRICS / "pp1_experiment_summary.md").write_text(summary_text, encoding="utf-8")
print("Notebook 05 PASSED. Saved evidence:")
for f in sorted(list(OUT_METRICS.iterdir())+list(OUT_FIGURES.iterdir())):
    print(" -",f.relative_to(ROOT))